# 13 — Election Geography Join Readiness

This notebook audits the ward/division geography in `ward_result_summary_v1.csv` and decides which result areas are ready to be crosswalked to OA21.

It does **not** allocate votes yet. It only answers:

- Does this result have a usable source geography code?
- Is the source geography a ward/electoral division or a county electoral division?
- Do we have a suitable OA21 lookup or CED bridge file?
- Which rows are ready for Notebook 14?

Expected input folders:

```text
C:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results
C:\Users\keena\Documents\Electoral_Tribes\data\geography
```


## 13.1 Project paths and expected files


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

ELECTION_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
GEOGRAPHY_DIR = PROJECT_DIR / "data" / "geography"
OUTPUT_DIR = ELECTION_DIR / "geography_readiness_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WARD_SUMMARY_PATH = ELECTION_DIR / "ward_result_summary_v2_with_2023_matches.csv"
CANDIDATE_PATH = ELECTION_DIR / "local_election_results_raw_v1.csv"

GEOGRAPHY_FILES = {
    2021: {
        "oa_to_ward": "oa11_to_wd21_lad21_eng_wal.csv",  # kept for inventory only; not direct OA21-compatible
        "wd_to_ced": "wd21_to_lad21_ct21_ced21_eng.xlsx",
        "oa21_compatible": False,
    },
    2022: {
        "oa_to_ward": "oa21_to_wd22_lad22_ctyua22_rgn22_ctry22_eng_wal.csv",
        "wd_to_ced": "wd22_to_lad22_cty22_ced22_eng.csv",
        "oa21_compatible": True,
    },
    2023: {
        "oa_to_ward": "oa21_to_wd23_lad23_eng_wal.csv",
        "wd_to_ced": "wd23_to_lad23_cty23_ced23_eng.csv",
        "oa21_compatible": True,
    },
    2024: {
        "oa_to_ward": "oa21_to_wd24_lad24_eng_wal.csv",
        "wd_to_ced": "wd24_to_lad24_cty24_ced24_eng.csv",
        "oa21_compatible": True,
    },
    2025: {
        "oa_to_ward": "oa21_to_wd25_lad25_eng_wal(may25).csv",
        "wd_to_ced": "wd25_to_lad25_cty25_ced25_eng.csv",
        "oa21_compatible": True,
    },
}

print("Project directory:", PROJECT_DIR)
print("Election directory:", ELECTION_DIR)
print("Geography directory:", GEOGRAPHY_DIR)
print("Output directory:", OUTPUT_DIR)

if not WARD_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Ward summary not found: {WARD_SUMMARY_PATH}. Run Notebook 12 first.")


Project directory: c:\Users\keena\Documents\Electoral_Tribes
Election directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results
Geography directory: c:\Users\keena\Documents\Electoral_Tribes\data\geography
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1


## 13.2 Helper functions

These functions normalise column names, detect useful geography columns in lookup files, and classify result-area codes.


In [14]:
def clean_colname(col):
    return str(col).strip()


def norm_key(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    value = value.replace("&", "AND")
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalise_code(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    if value in ["", "NAN", "NONE", "NULL"]:
        return pd.NA
    return value


def classify_geography_code(code):
    code = normalise_code(code)
    if pd.isna(code):
        return "name_only_no_ons_code"

    if str(code).startswith(("E58", "W58")):
        return "county_electoral_division"
    if str(code).startswith(("E05", "W05")):
        return "electoral_ward_or_division"
    if str(code).startswith(("E0", "W0")):
        return "other_ons_geography"

    return "unknown_code_type"


def read_lookup_file(path):
    if not path.exists():
        return None

    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, low_memory=False)

    df.columns = [clean_colname(c) for c in df.columns]
    return df


def find_first_col(df, candidates):
    cols = {c.upper(): c for c in df.columns}
    for cand in candidates:
        if cand.upper() in cols:
            return cols[cand.upper()]
    return None


def code_set_from_lookup(df, year, kind="ward"):
    if df is None:
        return set(), None

    if kind == "ward":
        candidates = [f"WD{str(year)[-2:]}CD", "WDCD", "Ward code", "Ward/ED code", "WARDCODE"]
    elif kind == "ced":
        candidates = [f"CED{str(year)[-2:]}CD", "CEDCD", "County electoral division code", "County Electoral Division code"]
    else:
        candidates = []

    col = find_first_col(df, candidates)
    if col is None:
        # Fallback: search any column that looks like the requested code family.
        prefix = "CED" if kind == "ced" else "WD"
        possible = [c for c in df.columns if c.upper().startswith(prefix) and c.upper().endswith("CD")]
        col = possible[0] if possible else None

    if col is None:
        return set(), None

    return set(df[col].dropna().astype(str).str.strip().str.upper()), col


## 13.3 Load ward result summary and geography lookup inventory


In [15]:
ward_summary = pd.read_csv(WARD_SUMMARY_PATH, low_memory=False)

if "ward_code" not in ward_summary.columns:
    raise ValueError("ward_result_summary_v1.csv must contain a ward_code column.")

ward_summary["source_geography_code"] = ward_summary["ward_code"].map(normalise_code)
ward_summary["source_geography_name"] = ward_summary.get("standard_ward_name", ward_summary.get("ward_name", pd.NA))
ward_summary["source_boundary_year"] = pd.to_numeric(ward_summary.get("boundary_year", ward_summary.get("source_year")),errors="coerce").astype("Int64")

# Recompute geography type from the code as a safety check.
ward_summary["detected_geography_type"] = ward_summary["source_geography_code"].map(classify_geography_code)

lookup_inventory = []
oa_ward_code_sets = {}
ced_code_sets = {}

for year, spec in GEOGRAPHY_FILES.items():
    oa_path = GEOGRAPHY_DIR / spec["oa_to_ward"]
    ced_path = GEOGRAPHY_DIR / spec["wd_to_ced"]

    oa_df = read_lookup_file(oa_path)
    ced_df = read_lookup_file(ced_path)

    ward_codes, ward_code_col = code_set_from_lookup(oa_df, year, kind="ward")
    ced_codes, ced_code_col = code_set_from_lookup(ced_df, year, kind="ced")

    oa_ward_code_sets[year] = ward_codes
    ced_code_sets[year] = ced_codes

    lookup_inventory.append({
        "year": year,
        "lookup_type": "oa_to_ward",
        "filename": spec["oa_to_ward"],
        "path_exists": oa_path.exists(),
        "oa21_compatible": spec["oa21_compatible"],
        "detected_code_column": ward_code_col,
        "unique_codes": len(ward_codes),
    })

    lookup_inventory.append({
        "year": year,
        "lookup_type": "wd_to_ced",
        "filename": spec["wd_to_ced"],
        "path_exists": ced_path.exists(),
        "oa21_compatible": spec["oa21_compatible"],
        "detected_code_column": ced_code_col,
        "unique_codes": len(ced_codes),
    })

lookup_inventory = pd.DataFrame(lookup_inventory)
lookup_inventory.to_csv(OUTPUT_DIR / "election_geography_lookup_inventory_v1.csv", index=False)

display(lookup_inventory)


,year,lookup_type,filename,path_exists,oa21_compatible,detected_code_column,unique_codes
0,2021,oa_to_ward,oa11_to_wd21_lad21_eng_wal.csv,True,False,WD21CD,7860
1,2021,wd_to_ced,wd21_to_lad21_ct21_ced21_eng.xlsx,True,False,CED21CD,1574
2,2022,oa_to_ward,oa21_to_wd22_lad22_ctyua22_rgn22_ctry22_eng_wa...,True,True,WD22CD,7638
3,2022,wd_to_ced,wd22_to_lad22_cty22_ced22_eng.csv,True,True,CED22CD,1574
4,2023,oa_to_ward,oa21_to_wd23_lad23_eng_wal.csv,True,True,WD23CD,7608
5,2023,wd_to_ced,wd23_to_lad23_cty23_ced23_eng.csv,True,True,CED23CD,1367
6,2024,oa_to_ward,oa21_to_wd24_lad24_eng_wal.csv,True,True,WD24CD,7563
7,2024,wd_to_ced,wd24_to_lad24_cty24_ced24_eng.csv,True,True,CED24CD,1366
8,2025,oa_to_ward,oa21_to_wd25_lad25_eng_wal(may25).csv,True,True,WD25CD,7572
9,2025,wd_to_ced,wd25_to_lad25_cty25_ced25_eng.csv,True,True,CED25CD,1382


## 13.4 Classify join readiness

The rule is deliberately conservative:

- 2021 is ingested but **not OA21-ready** because the available lookup is OA11-based.
- 2022–2025 ward results are ready if their source ward code exists in the matching OA21→WDxx lookup.
- County electoral divisions are ready only if a matching `CEDxxCD` can be found in the WD→CED file. The actual OA allocation happens in Notebook 14.
- Name-only rows are exported for dictionary/manual review.


In [17]:
# ============================================================
# 13.4 Assess join readiness
# ============================================================
#
# This cell classifies each ward/division result area according to whether it
# can currently be crosswalked to OA21 using the geography files available.
#
# Important:
# If this notebook is rerun, or if ward_result_summary_v2 already contains
# older readiness columns, we remove them first to avoid duplicate column names.
# ============================================================

# Remove any previous readiness columns before recalculating
readiness_cols = [
    "atlas_join_ready",
    "join_readiness_status",
    "recommended_next_action",
    "source_code_found_in_lookup",
]

ward_summary_clean = ward_summary.drop(
    columns=[c for c in readiness_cols if c in ward_summary.columns],
    errors="ignore"
).copy()


def assess_join_readiness(row):
    year = row.get("source_boundary_year")
    code = normalise_code(row.get("source_geography_code"))
    detected_type = row.get("detected_geography_type")

    if pd.isna(year):
        return pd.Series({
            "atlas_join_ready": False,
            "join_readiness_status": "missing_boundary_year",
            "recommended_next_action": "manual_review_boundary_year",
            "source_code_found_in_lookup": False,
        })

    year = int(year)

    if year == 2021:
        return pd.Series({
            "atlas_join_ready": False,
            "join_readiness_status": "not_oa21_ready_oa11_lookup_only",
            "recommended_next_action": "keep_for_history_build_oa11_to_oa21_crosswalk_later",
            "source_code_found_in_lookup": False,
        })

    if detected_type == "name_only_no_ons_code":
        return pd.Series({
            "atlas_join_ready": False,
            "join_readiness_status": "name_match_required",
            "recommended_next_action": "review_ward_name_dictionary",
            "source_code_found_in_lookup": False,
        })

    if detected_type == "electoral_ward_or_division":
        found = code in oa_ward_code_sets.get(year, set())
        return pd.Series({
            "atlas_join_ready": bool(found),
            "join_readiness_status": "ward_oa21_lookup_available" if found else "ward_code_not_found_in_oa21_lookup",
            "recommended_next_action": "send_to_notebook_14" if found else "check_ward_code_or_boundary_year",
            "source_code_found_in_lookup": bool(found),
        })

    if detected_type == "county_electoral_division":
        found = code in ced_code_sets.get(year, set())
        return pd.Series({
            "atlas_join_ready": bool(found),
            "join_readiness_status": "ced_bridge_available" if found else "ced_code_not_found_in_wd_to_ced_lookup",
            "recommended_next_action": "send_to_notebook_14_ced_bridge" if found else "check_ced_lookup_or_source_code",
            "source_code_found_in_lookup": bool(found),
        })

    return pd.Series({
        "atlas_join_ready": False,
        "join_readiness_status": "unknown_code_type",
        "recommended_next_action": "manual_review_code_type",
        "source_code_found_in_lookup": False,
    })


readiness_assessment = ward_summary_clean.apply(assess_join_readiness, axis=1)

readiness = pd.concat(
    [ward_summary_clean, readiness_assessment],
    axis=1
)

# Guard against duplicate columns
duplicate_columns = readiness.columns[readiness.columns.duplicated()].tolist()

if duplicate_columns:
    raise ValueError(f"Duplicate columns remain after readiness assessment: {duplicate_columns}")

readiness_summary = (
    readiness
    .groupby(
        [
            "source_year",
            "source_boundary_year",
            "detected_geography_type",
            "join_readiness_status",
        ],
        dropna=False,
        as_index=False
    )
    .agg(
        result_areas=("result_area_key", "nunique"),
        rows=("result_area_key", "count"),
    )
    .sort_values(["source_year", "join_readiness_status"])
)

display(readiness_summary)

,source_year,source_boundary_year,detected_geography_type,join_readiness_status,result_areas,rows
0,2021,2021,county_electoral_division,not_oa21_ready_oa11_lookup_only,1366,1366
1,2021,2021,electoral_ward_or_division,not_oa21_ready_oa11_lookup_only,2498,2498
2,2022,2022,electoral_ward_or_division,ward_code_not_found_in_oa21_lookup,1,1
3,2022,2022,electoral_ward_or_division,ward_oa21_lookup_available,3609,3609
4,2023,2023,electoral_ward_or_division,ward_oa21_lookup_available,4831,4831
5,2024,2024,electoral_ward_or_division,ward_oa21_lookup_available,1903,1903
6,2025,2025,county_electoral_division,ced_bridge_available,888,888
7,2025,2025,electoral_ward_or_division,ward_oa21_lookup_available,513,513


## 13.5 Export readiness files


In [18]:
readiness_path = OUTPUT_DIR / "election_geography_join_readiness_v1.csv"
ready_path = OUTPUT_DIR / "election_areas_ready_for_oa21_crosswalk_v1.csv"
not_ready_path = OUTPUT_DIR / "election_areas_not_ready_for_oa21_crosswalk_v1.csv"
name_review_path = OUTPUT_DIR / "election_name_match_required_v1.csv"
summary_path = OUTPUT_DIR / "election_geography_join_readiness_summary_v1.csv"

readiness.to_csv(readiness_path, index=False)
readiness[readiness["atlas_join_ready"]].to_csv(ready_path, index=False)
readiness[~readiness["atlas_join_ready"]].to_csv(not_ready_path, index=False)
readiness[readiness["join_readiness_status"].eq("name_match_required")].to_csv(name_review_path, index=False)
readiness_summary.to_csv(summary_path, index=False)

print("Saved:")
print(" ", readiness_path)
print(" ", ready_path)
print(" ", not_ready_path)
print(" ", name_review_path)
print(" ", summary_path)


Saved:
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1\election_geography_join_readiness_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1\election_areas_ready_for_oa21_crosswalk_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1\election_areas_not_ready_for_oa21_crosswalk_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1\election_name_match_required_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\geography_readiness_v1\election_geography_join_readiness_summary_v1.csv


## 13.6 What to check before moving to Notebook 14

Proceed to Notebook 14 when:

1. The 2022–2025 ward rows you care about mostly show `ward_oa21_lookup_available`.
2. Any 2023 name-only rows have either been accepted as not-ready or corrected in the ward dictionary and rerun through Notebooks 11–12.
3. County electoral division rows show `ced_bridge_available` where relevant.

Rows from 2021 are expected to remain not-ready for the OA21 atlas unless you later build an OA11→OA21 historical bridge.
